In [ ]:
import requests
from PIL import Image
import io
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import json
from collections import Counter
import importlib
import utils
importlib.reload(utils)
from utils import segementation_metrics,accuracy_score,detection_score
import random
import os
from pathlib import Path
import cv2
from ultralytics import YOLO 

image_folder=r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_referances\data.yaml"
model = YOLO(r'C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_tools\best.pt')
class FoodDetectionReporter:
    def __init__(self):
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
    def calculate_iou(self, mask1: np.ndarray, mask2: np.ndarray) -> float:
        """Calculate IoU between two binary masks"""
        intersection = np.logical_and(mask1, mask2).sum()
        union = np.logical_or(mask1, mask2).sum()
        return intersection / union if union > 0 else 0

    def calculate_segmentation_metrics(self, pred_masks: List[np.ndarray], 
                                    gt_masks: List[np.ndarray]) -> Dict:
        """Calculate segmentation metrics including IoU"""
        metrics = {
            'iou_scores': [],
            'mean_iou': 0,
            'per_class_iou': {}
        }
        
        # Calculate IoU for each mask pair
        for pred_mask, gt_mask in zip(pred_masks, gt_masks):
            iou = self.calculate_iou(pred_mask, gt_mask)
            metrics['iou_scores'].append(iou)
            
        metrics['mean_iou'] = np.mean(metrics['iou_scores']) if metrics['iou_scores'] else 0
        return metrics

    def generate_iou_plot(self, iou_scores: List[float], labels: List[str]) -> str:
        """Generate IoU distribution plot"""
        plt.figure(figsize=(10, 5))
        plt.bar(labels, iou_scores)
        plt.ylim(0, 1)
        plt.xticks(rotation=45, ha='right')
        plt.ylabel('IoU Score')
        plt.title('Segmentation IoU Scores by Food Item')
        
        output_dir = 'reports'
        os.makedirs(output_dir, exist_ok=True)
        plot_path = os.path.join(output_dir, f'iou_plot_{self.timestamp}.png')
        # plt.tight_layout()
        # plt.savefig(plot_path)
        # plt.close()
        return plot_path
    def analyze_confidence_scores(self, detections: List[Dict]) -> Dict:
        """Analyze confidence scores from detections"""
        confidence_scores = [d['confidence'] for d in detections]
        return {
            'mean': np.mean(confidence_scores),
            'median': np.median(confidence_scores),
            'min': np.min(confidence_scores),
            'max': np.max(confidence_scores),
            'std': np.std(confidence_scores)
        }

    def analyze_class_distribution(self, detections: List[Dict]) -> Dict:
        """Analyze distribution of detected classes"""
        labels = [d['label'] for d in detections]
        class_counts = Counter(labels)
        return dict(class_counts)

    def generate_confidence_plot(self, detections: List[Dict]) -> str:
        """Generate confidence score distribution plot
        
        Args:
            detections: List of detection results with confidence scores
            
        Returns:
            str: Path to saved plot image
        """
        confidence_scores = [d['confidence'] for d in detections]
        labels = [d['label'] for d in detections]
        
        # plt.figure(figsize=(10, 5))
        # plt.bar(labels, confidence_scores)
        # plt.ylim(0, 1)
        # plt.xticks(rotation=45, ha='right')
        # plt.ylabel('Confidence Score')
        # plt.title('Detection Confidence Scores by Food Item')
        
        output_dir = 'reports'
        os.makedirs(output_dir, exist_ok=True)
        plot_path = os.path.join(output_dir, f'confidence_plot_{self.timestamp}.png')
        # plt.tight_layout()
        # plt.savefig(plot_path)
        # plt.close()
        
        return plot_path

    def generate_segmentation_visualization(self, image: np.ndarray, 
                                         masks: List[np.ndarray]) -> str:
        """Generate visualization of segmentation masks"""
        # Create a colored visualization of all masks
        vis_image = image.copy()
        colors = [(255,0,0), (0,255,0), (0,0,255), (255,255,0), 
                 (255,0,255), (0,255,255)]  # Add more colors if needed
        
        for mask, color in zip(masks, colors):
            vis_image[mask > 0] = color
            
        output_dir = 'reports'
        vis_path = os.path.join(output_dir, f'segmentation_vis_{self.timestamp}.png')
        # cv2.imwrite(vis_path, cv2.cvtColor(vis_image, cv2.COLOR_RGB2BGR))
        return vis_path

    def load_image(self, image_path: str) -> Optional[Image.Image]:
        """Load image from URL or local path"""
        try:
            if image_path.startswith(('http://', 'https://')):
                response = requests.get(image_path)
                response.raise_for_status()
                return Image.open(io.BytesIO(response.content))
            else:
                image_path = os.path.abspath(image_path)
                if not os.path.exists(image_path):
                    raise FileNotFoundError(f"Image file not found: {image_path}")
                return Image.open(image_path)
        except Exception as e:
            print(f"Error loading image: {e}")
            return None

    
        
        output_dir = 'reports'
        os.makedirs(output_dir, exist_ok=True)
        report_path = os.path.join(output_dir, f'detection_report_{self.timestamp}.html')
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        return report_path
    def generate_json_report(self, image_path: str, detections: List[Dict], 
                           confidence_plot_path: str, segmentation_metrics: Dict,
                           iou_plot_path: str, segmentation_vis_path: str) -> str:
        """Generate JSON report with analysis results including segmentation metrics
        
        Args:
            image_path: Path to input image
            detections: List of detection results
            confidence_plot_path: Path to confidence plot image
            segmentation_metrics: Dictionary of segmentation metrics
            iou_plot_path: Path to IoU plot image
            segmentation_vis_path: Path to segmentation visualization image
            
        Returns:
            str: Path to saved JSON report
        """
        confidence_stats = self.analyze_confidence_scores(detections)
        class_distribution = self.analyze_class_distribution(detections)
        
        # Convert paths to relative paths
        relative_image_path = os.path.relpath(image_path) if not image_path.startswith(('http://', 'https://')) else image_path
        relative_confidence_path = os.path.relpath(confidence_plot_path)
        relative_iou_plot_path = os.path.relpath(iou_plot_path)
        relative_seg_vis_path = os.path.relpath(segmentation_vis_path)
        segmentation_metric=segementation_metrics()
        accuracy_class= accuracy_score()
        # Create report dictionary
        report_data = {
            "metadata": {
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                # "image_path": relative_image_path
            },
            "detection_results": {
                # "detections": [
                #     {
                #         "label": detection["label"],
                #         "confidence": detection["confidence"]
                #     } for detection in detections
                # ],
                "confidence_statistics": confidence_stats,
                "class_distribution": class_distribution,
                # "confidence_plot_path": relative_confidence_path,
                "Detection Results":f'Menu Labeling accuracy: {detection_score(model,image_folder) * 100:.2f}%'
            },
            "Menu_segmentation_results": {
                "metrics": {

                    # "mean_iou": float(segmentation_metric["mean_iou"][0]),
                    # "best_iou": float(segmentation_metric["best_iou"][0]),
                    # "worst_iou": float(segmentation_metric["worst_iou"][0]),
                    "segementation_metrics_accuracy": f'Menu Seperation accuracy: {float(segmentation_metric["mean_iou"][0]) * 100:.2f}%'

                    
                    # "per_class_iou": {
                    #     detection["label"]: iou_score
                    #     for detection, iou_score in zip(detections, segmentation_metrics["iou_scores"])
                    # }
                },
                # "visualization_paths": {
                #     "iou_plot": relative_iou_plot_path,
                #     "segmentation_visualization": relative_seg_vis_path
                # },
                "Menu_Recognition_Accuracy": {
                    "accuracy":f'Menu Recogniton Accuracy: {accuracy_class * 100:.2f}%'}
                

                
            }
        }
        
        # Save report
        output_dir = 'reports'
        os.makedirs(output_dir, exist_ok=True)
        report_path = os.path.join(output_dir, f'detection_report_{self.timestamp}.json')
        
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(report_data, f, indent=4)
        
        return report_path
def generate_food_detection_report(image_path: str, detections: List[Dict], true_label:List[Dict],
                                 pred_masks: List[np.ndarray], 
                                 gt_masks: List[np.ndarray], 
                                 report_format: str = 'json') -> str:
    """
    Generate food detection report with segmentation metrics
    
    Args:
        image_path: Path to image file or URL
        detections: List of detection results with confidence scores
        pred_masks: List of predicted segmentation masks
        gt_masks: List of ground truth segmentation masks
        report_format: Output format ('json' or 'html'), defaults to 'json'
    """
    reporter = FoodDetectionReporter()
    
    # Load image
    image = reporter.load_image(image_path)
    if image is None:
        return "Error: Could not load image"
    
    # Convert image to numpy array
    image_np = np.array(image)
    
    # Calculate segmentation metrics
    segmentation_metrics = reporter.calculate_segmentation_metrics(pred_masks, gt_masks)
    
    # Generate plots and visualizations
    confidence_plot_path = reporter.generate_confidence_plot(detections)
    iou_plot_path = reporter.generate_iou_plot(
        segmentation_metrics['iou_scores'], 
        [d['label'] for d in detections]
    )
    segmentation_vis_path = reporter.generate_segmentation_visualization(
        image_np, pred_masks
    )
    
    # Generate report based on format
    
    report_path = reporter.generate_json_report(
        image_path, detections, confidence_plot_path,
        segmentation_metrics, iou_plot_path, segmentation_vis_path
    )

    
    return report_path

# Example usage
if __name__ == "__main__":
    image_path = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_exemples\2-d.jpg"
    
    # Sample detection results
    detections = [
        {'label': 'Fried egg', 'confidence': 0.85},
        {'label': 'Beansprouts', 'confidence': 0.87},
        {'label': 'Cabbage kimchi', 'confidence': 0.81},
        {'label': 'Grilled offal', 'confidence': 0.82}
    ]

    True_labels=[
        {'label': 'Fried egg', 'confidence': 0.85},
        {'label': 'Beansprouts', 'confidence': 0.87},
        {'label': 'Cabbage kimchi', 'confidence': 0.81},
        {'label': 'Grilled offal', 'confidence': 0.82}
    ]
    
    # Create dummy masks for example
    # In practice, these would come from your segmentation model
    image = np.array(Image.open(image_path))
    height, width = image.shape[:2]
    pred_masks = [np.zeros((height, width), dtype=bool) for _ in detections]
    gt_masks = [np.zeros((height, width), dtype=bool) for _ in detections]
    
    report_path = generate_food_detection_report(
        image_path, detections, True_labels,pred_masks, gt_masks,
    )
    print(f"Report generated: {report_path}")

Ultralytics 8.3.8  Python-3.12.7 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLOv8s summary (fused): 186 layers, 9,828,051 parameters, 0 gradients, 23.3 GFLOPs


val: Scanning C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_referances\train\labels.cache... 21 images, 0 backgrounds, 1 corrupt: 100%|██████████| 21/21 [00:00<?, ?it/s]

val: WARNING  C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_referances\train\images\2-a_jpg.rf.44ef4280b0f75a4ac5ee9f2dbfd7f2bd.jpg: ignoring corrupt image/label: Label class 4 exceeds dataset class count 1. Possible class labels are 0-0
WARNING  Box and segment counts should be equal, but got len(segments) = 185, len(boxes) = 190. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.60s/it]


                   all         20        190      0.978      0.505      0.564      0.381
Speed: 2.6ms preprocess, 19.5ms inference, 0.0ms loss, 5.4ms postprocess per image
Results saved to runs\detect\val4
Report generated: reports\detection_report_20241115_083350.json


test for all the folder